<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Audits course-difficulty fallback levels across prepared model splits.

**Notebook Shape:** 9 cells (8 code, 1 markdown).

**Inputs / Data Sources:**
- `df_train = pd.read_parquet(TRAIN_PATH)`
- `df_valid = pd.read_parquet(VALID_PATH)`
- `df_test = pd.read_parquet(TEST_PATH)`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Load enriched train/valid/test data.
2. Count fallback levels and missing difficulty flags.
3. Inspect examples where support is low or global priors were used.

**Maintainability Notes:** Diagnostic thresholds should be captured in tests if fallback behaviour becomes a model-quality gate.


# Course difficulty fallback diagnostic

Read-only investigation of why validation/test rows fall to the global course difficulty fallback.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd

from src.paths import MODEL_DATA_DIR

print('Setup: loading saved splits without rebuilding them')
print(f'Using MODEL_DATA_DIR: {MODEL_DATA_DIR}')

TRAIN_PATH = MODEL_DATA_DIR / "df_train_difficulty.parquet"
VALID_PATH = MODEL_DATA_DIR / "df_valid_difficulty.parquet"
TEST_PATH = MODEL_DATA_DIR / "df_test_difficulty.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_valid = pd.read_parquet(VALID_PATH)
df_test = pd.read_parquet(TEST_PATH)

print(f'df_train: {df_train.shape}')
print(f'df_valid: {df_valid.shape}')
print(f'df_test : {df_test.shape}')

Setup: loading saved splits without rebuilding them
Using MODEL_DATA_DIR: D:\AI\Real projects\Academic_Advisor\data\model_data
df_train: (450465, 70)
df_valid: (156097, 70)
df_test : (110008, 70)


In [2]:
print('Shared setup: derive Level-1 and Level-2 train lookup sets')

def course_id_from_degree_course_key(series):
    return series.astype('string').str.rsplit('__', n=1).str[-1]

train_degree_course_keys = set(df_train['degree_course_key'].astype('string').dropna())
train_course_ids_from_key = course_id_from_degree_course_key(df_train['degree_course_key'])
train_course_ids = set(train_course_ids_from_key.dropna())

train_course_to_degrees = (
    df_train
    .assign(_course_id_from_key=train_course_ids_from_key)
    .dropna(subset=['_course_id_from_key', 'degree_id'])
    .groupby('_course_id_from_key')['degree_id']
    .agg(lambda s: ', '.join(map(str, sorted(pd.unique(s.astype('string'))))))
)

print(f'Level-1 train degree_course_key count: {len(train_degree_course_keys):,}')
print(f'Level-2 train course_id count        : {len(train_course_ids):,}')


Shared setup: derive Level-1 and Level-2 train lookup sets
Level-1 train degree_course_key count: 1,666
Level-2 train course_id count        : 811


In [3]:
print('Check 1: recompute test rows that miss Level 1 and Level 2')

test_course_id_from_key = course_id_from_degree_course_key(df_test['degree_course_key'])
test_l1_hit = df_test['degree_course_key'].astype('string').isin(train_degree_course_keys)
test_l2_hit = test_course_id_from_key.isin(train_course_ids)

test_global_mask = (~test_l1_hit) & (~test_l2_hit)
test_global_fallback_rows = df_test.loc[test_global_mask].copy()
test_global_fallback_rows['_course_id_from_key'] = test_course_id_from_key.loc[test_global_mask].values

test_global_count = int(test_global_mask.sum())
test_global_pct = test_global_count / len(df_test) * 100
print(f'Recomputed test global fallback rows: {test_global_count:,} of {len(df_test):,} ({test_global_pct:.2f}%)')

if 'course_difficulty_fallback_level' in df_test.columns:
    saved_l3_count = int(df_test['course_difficulty_fallback_level'].eq(3).sum())
    saved_l3_pct = saved_l3_count / len(df_test) * 100
    print(f'Saved fallback level 3 rows        : {saved_l3_count:,} of {len(df_test):,} ({saved_l3_pct:.2f}%)')
    print(f'Matches saved fallback level 3?    : {test_global_count == saved_l3_count}')
else:
    print('Saved fallback column not present; only recomputed count is available.')


Check 1: recompute test rows that miss Level 1 and Level 2
Recomputed test global fallback rows: 34,699 of 110,008 (31.54%)
Saved fallback column not present; only recomputed count is available.


In [4]:
print('Check 2: split test global fallback course_ids into seen vs fully new')

test_global_course_ids = pd.Index(test_global_fallback_rows['_course_id_from_key'].dropna().unique())
test_global_course_ids_in_train = test_global_course_ids[test_global_course_ids.isin(train_course_ids)]
test_global_course_ids_new = test_global_course_ids[~test_global_course_ids.isin(train_course_ids)]

print(f'Unique course_id values in test_global_fallback_rows: {len(test_global_course_ids):,}')
print(f'  appear anywhere in df_train under any degree : {len(test_global_course_ids_in_train):,}')
print(f'  never appear in df_train under any degree    : {len(test_global_course_ids_new):,}')


Check 2: split test global fallback course_ids into seen vs fully new
Unique course_id values in test_global_fallback_rows: 273
  appear anywhere in df_train under any degree : 0
  never appear in df_train under any degree    : 273


In [5]:
print('Check 3: inspect completely new test course_ids')

test_fully_new_rows = test_global_fallback_rows.loc[
    test_global_fallback_rows['_course_id_from_key'].isin(test_global_course_ids_new)
].copy()

print('Distribution across degree_id:')
print(test_fully_new_rows['degree_id'].value_counts(dropna=False).to_string())

print('\nDistribution across part_year:')
print(test_fully_new_rows['part_year'].value_counts(dropna=False).sort_index().to_string())

sample_cols = ['_course_id_from_key', 'degree_id', 'part_year', 'course_credits', 'requirement_type_id']
sample_new_test_courses = (
    test_fully_new_rows[sample_cols]
    .drop_duplicates('_course_id_from_key')
    .rename(columns={'_course_id_from_key': 'course_id'})
    .sort_values(['part_year', 'degree_id', 'course_id'])
    .head(15)
)

print('\nSample of 15 completely new test course_id values:')
print(sample_new_test_courses.to_string(index=False))


Check 3: inspect completely new test course_ids
Distribution across degree_id:
degree_id
26.111    3594
40.111    3576
41.111    2612
29.111    2175
47.111    2146
42.111    1917
46.111    1862
48.111    1829
27.111    1796
45.111    1542
44.111    1459
55.111     951
34.111     830
30.111     807
36.111     798
53.111     795
49.111     786
7.111      583
31.111     573
50.111     475
65.111     427
59.111     305
58.111     289
35.111     269
33.111     264
56.111     264
57.111     260
54.111     248
64.111     240
60.111     225
37.111     208
52.111     206
61.111     198
39.111     166
4.111       14
6.111        9
5.111        1

Distribution across part_year:
part_year
2024    22132
2025    12567

Sample of 15 completely new test course_id values:
course_id degree_id  part_year  course_credits  requirement_type_id
 1163.111    26.111       2024             2.0                    3
 1167.111    26.111       2024             2.0                    4
 1169.111    26.111       2024

In [6]:
print('Check 4: test Level-2 rows where course_id exists in train but degree_course_key is new')

test_level2_diff_degree_mask = (~test_l1_hit) & test_l2_hit
test_level2_diff_degree_rows = df_test.loc[test_level2_diff_degree_mask].copy()
test_level2_diff_degree_rows['_course_id_from_key'] = test_course_id_from_key.loc[test_level2_diff_degree_mask].values
test_level2_diff_degree_rows['train_degree_ids_for_course_id'] = (
    test_level2_diff_degree_rows['_course_id_from_key'].map(train_course_to_degrees)
)

print(f'Test rows with Level-2 course_id match but no Level-1 key match: {len(test_level2_diff_degree_rows):,}')

sample_level2 = (
    test_level2_diff_degree_rows[
        ['degree_course_key', 'degree_id', '_course_id_from_key', 'part_year', 'train_degree_ids_for_course_id']
    ]
    .drop_duplicates(['degree_course_key', '_course_id_from_key'])
    .rename(columns={'_course_id_from_key': 'course_id'})
    .sort_values(['part_year', 'degree_course_key'])
    .head(10)
)

print('\nSample of 10 Level-2 test rows:')
print(sample_level2.to_string(index=False))


Check 4: test Level-2 rows where course_id exists in train but degree_course_key is new
Test rows with Level-2 course_id match but no Level-1 key match: 25,640

Sample of 10 Level-2 test rows:
degree_course_key degree_id course_id  part_year                                                                                         train_degree_ids_for_course_id
 26.111__1015.111    26.111  1015.111       2024        1.111, 10.111, 11.111, 13.111, 15.111, 18.111, 2.111, 21.111, 22.111, 24.111, 3.111, 4.111, 5.111, 6.111, 8.111
 26.111__1016.111    26.111  1016.111       2024         1.111, 10.111, 11.111, 13.111, 15.111, 2.111, 21.111, 22.111, 24.111, 3.111, 4.111, 5.111, 6.111, 7.111, 8.111
 26.111__1019.111    26.111  1019.111       2024         1.111, 10.111, 11.111, 13.111, 18.111, 2.111, 21.111, 22.111, 24.111, 3.111, 4.111, 5.111, 6.111, 7.111, 8.111
 26.111__1021.111    26.111  1021.111       2024                 1.111, 10.111, 11.111, 13.111, 2.111, 21.111, 22.111, 24.111, 3.111, 4

In [7]:
print('Check 5: same-shape quick comparison for valid')

valid_course_id_from_key = course_id_from_degree_course_key(df_valid['degree_course_key'])
valid_l1_hit = df_valid['degree_course_key'].astype('string').isin(train_degree_course_keys)
valid_l2_hit = valid_course_id_from_key.isin(train_course_ids)
valid_global_mask = (~valid_l1_hit) & (~valid_l2_hit)

valid_global_fallback_rows = df_valid.loc[valid_global_mask].copy()
valid_global_fallback_rows['_course_id_from_key'] = valid_course_id_from_key.loc[valid_global_mask].values

valid_global_count = int(valid_global_mask.sum())
valid_global_pct = valid_global_count / len(df_valid) * 100
print(f'Recomputed valid global fallback rows: {valid_global_count:,} of {len(df_valid):,} ({valid_global_pct:.2f}%)')

if 'course_difficulty_fallback_level' in df_valid.columns:
    saved_valid_l3_count = int(df_valid['course_difficulty_fallback_level'].eq(3).sum())
    saved_valid_l3_pct = saved_valid_l3_count / len(df_valid) * 100
    print(f'Saved valid fallback level 3 rows : {saved_valid_l3_count:,} of {len(df_valid):,} ({saved_valid_l3_pct:.2f}%)')
    print(f'Matches saved fallback level 3?    : {valid_global_count == saved_valid_l3_count}')

valid_global_course_ids = pd.Index(valid_global_fallback_rows['_course_id_from_key'].dropna().unique())
valid_global_course_ids_in_train = valid_global_course_ids[valid_global_course_ids.isin(train_course_ids)]
valid_global_course_ids_new = valid_global_course_ids[~valid_global_course_ids.isin(train_course_ids)]
print(f'Unique course_id values in valid global fallback rows: {len(valid_global_course_ids):,}')
print(f'  appear anywhere in df_train under any degree : {len(valid_global_course_ids_in_train):,}')
print(f'  never appear in df_train under any degree    : {len(valid_global_course_ids_new):,}')

valid_fully_new_rows = valid_global_fallback_rows.loc[
    valid_global_fallback_rows['_course_id_from_key'].isin(valid_global_course_ids_new)
].copy()

print('\nValid fully-new rows by degree_id:')
print(valid_fully_new_rows['degree_id'].value_counts(dropna=False).to_string())

print('\nValid fully-new rows by part_year:')
print(valid_fully_new_rows['part_year'].value_counts(dropna=False).sort_index().to_string())

sample_new_valid_courses = (
    valid_fully_new_rows[sample_cols]
    .drop_duplicates('_course_id_from_key')
    .rename(columns={'_course_id_from_key': 'course_id'})
    .sort_values(['part_year', 'degree_id', 'course_id'])
    .head(10)
)

print('\nSample of 10 completely new valid course_id values:')
print(sample_new_valid_courses.to_string(index=False))


Check 5: same-shape quick comparison for valid
Recomputed valid global fallback rows: 25,627 of 156,097 (16.42%)
Unique course_id values in valid global fallback rows: 182
  appear anywhere in df_train under any degree : 0
  never appear in df_train under any degree    : 182

Valid fully-new rows by degree_id:
degree_id
26.111    4834
29.111    2788
27.111    2537
40.111    1430
30.111    1101
34.111    1066
36.111    1061
41.111    1015
45.111     862
48.111     862
47.111     855
42.111     796
31.111     772
39.111     739
46.111     706
44.111     575
7.111      529
49.111     513
55.111     496
53.111     428
33.111     359
35.111     358
37.111     280
50.111     157
54.111     146
56.111     146
52.111     114
6.111       75
5.111       17
4.111       10

Valid fully-new rows by part_year:
part_year
2022     7972
2023    17655

Sample of 10 completely new valid course_id values:
course_id degree_id  part_year  course_credits  requirement_type_id
 1172.111    26.111       2022   

In [8]:
print('Check 6: final summary for test global fallback causes')

test_global_new_mask = test_global_fallback_rows['_course_id_from_key'].isin(test_global_course_ids_new)
test_global_new_rows = int(test_global_new_mask.sum())
test_global_other_rows = int(len(test_global_fallback_rows) - test_global_new_rows)

print(f'Test global-fallback rows, recomputed                 : {len(test_global_fallback_rows):,}')
print(f'  genuinely new course_id never seen in df_train      : {test_global_new_rows:,}')
print(f'  other / possible course_id matching problem         : {test_global_other_rows:,}')
print(f'Separate Level-2 rows: course seen, new degree pairing: {len(test_level2_diff_degree_rows):,}')

if test_global_other_rows > 0:
    print('FLAG: some global fallback rows have course_ids that appear in train; inspect parsing/join logic.')
else:
    print('No global fallback rows had a course_id that appears in train under the same Level-2 parsing rule.')


Check 6: final summary for test global fallback causes
Test global-fallback rows, recomputed                 : 34,699
  genuinely new course_id never seen in df_train      : 34,699
  other / possible course_id matching problem         : 0
Separate Level-2 rows: course seen, new degree pairing: 25,640
No global fallback rows had a course_id that appears in train under the same Level-2 parsing rule.
